# MentalBERT baseline — Google Colab runner

Fine-tunes the naive multi-class MentalBERT baseline (`src/modeling/train.py`) and
produces held-out softmax predictions (`src/modeling/predict.py`).

**Before you start:** set the runtime to a GPU — *Runtime → Change runtime type →
Hardware accelerator → GPU (T4 is fine)*.

**One-time Drive setup.** Put your data and outputs on Google Drive so they survive
runtime resets. Create this layout in *MyDrive* (upload your local `Datasets/` into it):
```
MyDrive/mental-health-fyp/
    Datasets/            <- upload your local Datasets/ here (DAIC-WOZ/, RedditMentalHealth/)
    Models/              <- created automatically for checkpoints/splits/predictions
```
The same `src/modeling` code runs here and locally — only the `--data-root` /
`--artifacts-root` paths differ.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this if you named your Drive folder differently.
DRIVE_ROOT = '/content/drive/MyDrive/mental-health-fyp'
DATA_ROOT = f'{DRIVE_ROOT}/Datasets'
ARTIFACTS_ROOT = f'{DRIVE_ROOT}/Models'

import os
assert os.path.isdir(DATA_ROOT), f'Datasets not found at {DATA_ROOT} — upload them to Drive first.'
print('Datasets found:', os.listdir(DATA_ROOT))

## 3. Clone the repo

Clones the code into the (ephemeral) Colab runtime. Re-running this cell in a fresh
session re-clones; the *data and models* live on Drive, so nothing is lost.

In [ ]:
%cd /content
![ -d mh-repo ] && rm -rf mh-repo
!git clone https://github.com/SoshanW/mental-health-screening-fyp.git mh-repo
%cd /content/mh-repo

## 4. Install dependencies

Colab ships torch (CUDA) preinstalled, so we install only the rest of the modeling stack.

In [ ]:
!pip install -q 'transformers>=4.40' 'accelerate>=0.26.0' 'scikit-learn>=1.4'
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 5. Sanity-check the harmonized data load

Optional but recommended — confirms the loaders see your Drive data before training.

In [ ]:
!python -m src.data --data-root "$DATA_ROOT"

## 6. (Optional) Fast wiring smoke-test

Runs the full train→predict pipeline in seconds on a tiny public model and 50 samples —
verifies the plumbing before committing to a real MentalBERT run. Skip once you trust it.

In [ ]:
!python -m src.modeling.train \
    --data-root "$DATA_ROOT" --artifacts-root "/content/smoke_models" \
    --model-name hf-internal-testing/tiny-random-bert \
    --max-train-samples 50 --num-epochs 1
!python -m src.modeling.predict --artifacts-root "/content/smoke_models"

## 7. Train the real MentalBERT baseline

Checkpoints, splits, and predictions are written under `Models/` **on Drive**, so they
persist. `--fp16` speeds things up on the GPU. Adjust `--batch-size` up (e.g. 32) if the
GPU has spare memory (T4 ≈ 15 GB).

In [ ]:
!python -m src.modeling.train \
    --data-root "$DATA_ROOT" --artifacts-root "$ARTIFACTS_ROOT" \
    --num-epochs 3 --batch-size 16 --fp16

## 8. Predict on the held-out test split

Reads `Models/splits/test.csv` and the trained checkpoint, writes
`Models/predictions/test_predictions.csv`, and prints accuracy + macro-F1.

In [ ]:
!python -m src.modeling.predict --artifacts-root "$ARTIFACTS_ROOT"

In [ ]:
import pandas as pd
pred = pd.read_csv(f'{ARTIFACTS_ROOT}/predictions/test_predictions.csv')
print('rows:', len(pred))
pred.head()